# Description: Supplementary Figure 01

This notebook exemplifies the issue of heteroscedasticity and how it could affect more straight forward computations of pBOLD

In [1]:
import pandas as pd
import numpy as np
import holoviews as hv
import panel as pn

import os.path as osp
from utils.basics import PRCS_DATA_DIR, ATLASES_DIR, TES_MSEC

ATLAS_NAME = 'Power264-discovery'
ATLAS_DIR = osp.join(ATLASES_DIR,ATLAS_NAME)

import hvplot.pandas
from nilearn.connectome import sym_matrix_to_vec

from utils.basics import echo_pairs_tuples, get_altas_info

echo_times_dict = TES_MSEC['discovery']

*** 
# 1. Load Parcellation Information

This is needed to plot the two connectivity matrices in panels (a) and (b)

In [2]:
roi_info_df, power264_nw_cmap = get_altas_info(ATLAS_DIR,ATLAS_NAME)
roi_idxs = roi_info_df.set_index(['ROI_Name', 'ROI_ID', 'Hemisphere', 'Network']).index

++ INFO [get_nw_cmap]: Gathering ROI information from file /data/SFIMJGC_HCP7T/BCBL2024/atlases/Power264-discovery/Power264-discovery.roi_info.csv
++ INFO: Number of ROIs = 226 | Number of Connections = 25425


***
# 2. Load Timeseries and FC_C across alll echoes for a representative scan

We will use a low motion and aggresively pre-processed scan for explanatory purposes.

In [3]:
sbj = 'MGSBJ05'
ses = 'constant_gated'
scenario = 'ALL_Tedana-fastica'

In [4]:
fc={}
for (e_x,e_y) in echo_pairs_tuples:
    roi_ts_path_x = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'errts.{sbj}.r01.{e_x}.volreg.spc.tproject_{scenario}.{ATLAS_NAME}_000.netts')
    roi_ts_x      = np.loadtxt(roi_ts_path_x)
    roi_ts_path_y = osp.join(PRCS_DATA_DIR,sbj,f'D03_Preproc_{ses}_NORDIC-off',f'errts.{sbj}.r01.{e_y}.volreg.spc.tproject_{scenario}.{ATLAS_NAME}_000.netts')
    roi_ts_y      = np.loadtxt(roi_ts_path_y)
    aux_ts_x = pd.DataFrame(roi_ts_x, columns=roi_info_df['ROI_Name'].values)
    aux_ts_y = pd.DataFrame(roi_ts_y, columns=roi_info_df['ROI_Name'].values)
    # Compute the full correlation matrix between aux_ts_x and aux_ts_y
    aux_c           = np.cov(aux_ts_x.T, aux_ts_y.T)[:aux_ts_x.shape[1], aux_ts_x.shape[1]:]
    fc[(e_x,e_y)]  = pd.DataFrame(aux_c,index=roi_idxs,columns=roi_idxs)

In [5]:
x_te1,x_te2    = 'e01','e02'
y_te1,y_te2    = 'e02','e03'

In [6]:
# Get Connectivity data: top triangle of the FC-C matrix
a = sym_matrix_to_vec(fc[(x_te1,x_te2)].values,discard_diagonal=True)
b = sym_matrix_to_vec(fc[(y_te1,y_te2)].values,discard_diagonal=True)
df = pd.DataFrame([a,b], index=['C(TEi,TEj)','C(TEk,TEl)']).T
x = df['C(TEi,TEj)'].values
y = df['C(TEk,TEl)'].values

df = pd.DataFrame([a,b], index=['C(TE1,TE2)','C(TE2,TE3)']).T

***
# 2. Create Suppl. Figure 01 - Panel A

Create lines marking the origin

In [7]:
hv.extension("matplotlib")
zero_point = hv.HLine(0).opts(linewidth=0.5, color='k', linestyle='dotted') * hv.VLine(0).opts(linewidth=0.5, color='k', linestyle='dotted')

<img src='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAYAAACqaXHeAAAABHNCSVQICAgIfAhkiAAAAAlwSFlz
AAAB+wAAAfsBxc2miwAAABl0RVh0U29mdHdhcmUAd3d3Lmlua3NjYXBlLm9yZ5vuPBoAAA6zSURB
VHic7ZtpeFRVmsf/5966taWqUlUJ2UioBBJiIBAwCZtog9IOgjqACsogKtqirT2ttt069nQ/zDzt
tI4+CrJIREFaFgWhBXpUNhHZQoKBkIUASchWla1S+3ar7r1nPkDaCAnZKoQP/D7mnPOe9/xy76n3
nFSAW9ziFoPFNED2LLK5wcyBDObkb8ZkxuaoSYlI6ZcOKq1eWFdedqNzGHQBk9RMEwFAASkk0Xw3
ETacDNi2vtvc7L0ROdw0AjoSotQVkKSvHQz/wRO1lScGModBFbDMaNRN1A4tUBCS3lk7BWhQkgpD
lG4852/+7DWr1R3uHAZVQDsbh6ZPN7CyxUrCzJMRouusj0ipRwD2uKm0Zn5d2dFwzX1TCGhnmdGo
G62Nna+isiUqhkzuKrkQaJlPEv5mFl2fvGg2t/VnzkEV8F5ioioOEWkLG86fvbpthynjdhXYZziQ
x1hC9J2NFyi8vCTt91Fh04KGip0AaG9zuCk2wQCVyoNU3Hjezee9bq92duzzTmxsRJoy+jEZZZYo
GTKJ6SJngdJqAfRzpze0+jHreUtPc7gpBLQnIYK6BYp/uGhw9YK688eu7v95ysgshcg9qSLMo3JC
4jqLKQFBgdKDPoQ+Pltb8dUyQLpeDjeVgI6EgLIQFT5tEl3rn2losHVsexbZ3EyT9wE1uGdkIPcy
BGxn8QUq1QrA5nqW5i2tLqvrrM9NK6AdkVIvL9E9bZL/oyfMVd/jqvc8LylzRBKDJSzIExwhQzuL
QYGQj4rHfFTc8mUdu3E7yoLtbTe9gI4EqVgVkug2i5+uXGo919ixbRog+3fTbQ8qJe4ZOYNfMoTI
OoshUNosgO60AisX15aeI2PSIp5KiFLI9ubb1vV3Qb2ltwLakUCDAkWX7/nHKRmmGIl9VgYsUhJm
2NXjKYADtM1ygne9QQDIXlk49FBstMKx66D1v4+XuQr7vqTe0VcBHQlRWiOCbmmSYe2SqtL6q5rJ
zsTb7lKx3FKOYC4DoqyS/B5bvLPxvD9Qtf6saxYLQGJErmDOdOMr/zo96km1nElr8bmPOBwI9COv
HnFPRIwmkSOv9kcAS4heRsidOkpeWBgZM+UBrTFAXNYL5Vf2ii9c1trNzpYdaoVil3WIc+wdk+gQ
noie3ecCcxt9ITcLAPWt/laGEO/9U6PmzZkenTtsSMQ8uYywJVW+grCstAvCIaAdArAsIWkRDDs/
KzLm2YcjY1Lv0UdW73HabE9n6V66cxSzfEmuJssTpKGVp+0vHq73FwL46eOjpMpbRAnNmJFrGJNu
Ukf9Yrz+3rghiumCKNXXWPhLYcjxGsIpoCMsIRoFITkW8AuyM8jC1+/QLx4bozCEJIq38+1rtpR6
V/yzb8eBlRb3fo5l783N0CWolAzJHaVNzkrTzlEp2bQ2q3TC5gn6wpnoQAmwSiGh2GitnTmVMc5O
UyfKWUKCIsU7+fZDKwqdT6DDpvkzAX4/+AMFjk0tDp5GRXLpQ2MUmhgDp5gxQT8+Y7hyPsMi8uxF
71H0oebujHALECjFKaW9Lm68n18wXp2kVzIcABytD5iXFzg+WVXkegpAsOOYziqo0OkK76GyquC3
ltZAzMhhqlSNmmWTE5T6e3IN05ITFLM4GdN0vtZ3ob8Jh1NAKXFbm5PtLU/eqTSlGjkNAJjdgn/N
aedXa0tdi7+t9G0FIF49rtMSEgAs1kDLkTPO7ebm4IUWeyh1bKomXqlgMG6kJmHcSM0clYLJ8XtR
1GTnbV3F6I5wCGikAb402npp1h1s7LQUZZSMIfALFOuL3UUrfnS8+rez7v9qcold5tilgHbO1fjK
9ubb17u9oshxzMiUBKXWqJNxd+fqb0tLVs4lILFnK71H0Ind7uiPgACVcFJlrb0tV6DzxqqTIhUM
CwDf1/rrVhTa33/3pGPxJYdQ2l2cbgVcQSosdx8uqnDtbGjh9SlDVSMNWhlnilfqZk42Th2ZpLpf
xrHec5e815zrr0dfBZSwzkZfqsv+1FS1KUknUwPARVvItfKUY+cn57yP7qv07UE3p8B2uhUwLk09
e0SCOrK+hbdYHYLjRIl71wWzv9jpEoeOHhGRrJAzyEyNiJuUqX0g2sBN5kGK6y2Blp5M3lsB9Qh4
y2Ja6x6+i0ucmKgwMATwhSjdUu49tKrQ/pvN5d53ml2CGwCmJipmKjgmyuaXzNeL2a0AkQ01Th5j
2DktO3Jyk8f9vcOBQHV94OK+fPumJmvQHxJoWkaKWq9Vs+yUsbq0zGT1I4RgeH2b5wef7+c7bl8F
eKgoHVVZa8ZPEORzR6sT1BzDUAD/d9F78e2Tzv99v8D+fLVTqAKAsbGamKey1Mt9Ann4eH3gTXTz
idWtAJ8PQWOk7NzSeQn/OTHDuEikVF1R4z8BQCy+6D1aWRfY0tTGG2OM8rRoPaeIj5ZHzJxszElN
VM8K8JS5WOfv8mzRnQAKoEhmt8gyPM4lU9SmBK1MCQBnW4KONT86v1hZ1PbwSXPw4JWussVjtH9Y
NCoiL9UoH/6PSu8jFrfY2t36erQHXLIEakMi1SydmzB31h3GGXFDFNPaK8Rme9B79Ixrd0WN+1ij
NRQ/doRmuFLBkHSTOm5GruG+pFjFdAmorG4IXH1Qua6ASniclfFtDYt+oUjKipPrCQB7QBQ2lrgP
fFzm+9XWUtcqJ3/5vDLDpJ79XHZk3u8nGZ42qlj1+ydtbxysCezrydp6ugmipNJ7WBPB5tydY0jP
HaVNzs3QzeE4ZpTbI+ZbnSFPbVOw9vsfnVvqWnirPyCNGD08IlqtYkh2hjZ5dErEQzoNm+6ykyOt
Lt5/PQEuSRRKo22VkydK+vvS1XEKlhCJAnsqvcVvH7f/ZU2R67eXbMEGAMiIV5oWZWiWvz5Fv2xG
sjqNJQRvn3Rs2lji/lNP19VjAQDgD7FHhujZB9OGqYxRkZxixgRDVlqS6uEOFaJUVu0rPFzctrnF
JqijImVp8dEKVWyUXDk92zAuMZ6bFwpBU1HrOw6AdhQgUooChb0+ItMbWJitSo5Ws3IAOGEOtL53
0vHZih9sC4vtofZ7Qu6523V/fmGcds1TY3V36pUsBwAbSlxnVh2xLfAD/IAIMDf7XYIkNmXfpp2l
18rkAJAy9HKFaIr/qULkeQQKy9zf1JgDB2uaeFNGijo5QsUyacNUUTOnGO42xSnv4oOwpDi1zYkc
efUc3I5Gk6PhyTuVKaOGyLUAYPGIoY9Pu/atL/L92+4q9wbflRJ2Trpm/jPjdBtfnqB/dIThcl8A
KG7hbRuKnb8qsQsVvVlTrwQAQMUlf3kwJI24Z4JhPMtcfng5GcH49GsrxJpGvvHIaeem2ma+KSjQ
lIwUdYyCY8j4dE1KzijNnIP2llF2wcXNnsoapw9XxsgYAl6k+KzUXbi2yP3KR2ecf6z3BFsBICdW
nvnIaG3eHybqX7vbpEqUMT+9OL4Qpe8VON7dXuFd39v19FoAABRVePbGGuXTszO0P7tu6lghUonE
llRdrhArLvmKdh9u29jcFiRRkfLUxBiFNiqSU9icoZQHo5mYBI1MBgBH6wMNb+U7Pnw337H4gi1Y
ciWs+uks3Z9fztUvfzxTm9Ne8XXkvQLHNytOOZeiD4e0PgkAIAYCYknKUNUDSXEKzdWNpnil7r4p
xqkjTarZMtk/K8TQ6Qve78qqvXurGwIJqcOUKfUWHsm8KGvxSP68YudXq4pcj39X49uOK2X142O0
Tz5/u/7TVybqH0rSya6ZBwD21/gubbrgWdDgEOx9W

Create line representing non-BOLD behavior (slope=1, intercept=0)

In [8]:
nonBOLD_line = hv.Slope(1,0).opts(linewidth=2, color='r', linestyle='dashed')

Create line representing BOLD behavior (slope=f(echo times), intercept=0)

In [9]:
BOLD_slope = (echo_times_dict[y_te1]*echo_times_dict[y_te2])/(echo_times_dict[x_te1]*echo_times_dict[x_te2])
BOLD_line = hv.Slope(BOLD_slope,0).opts(linewidth=2, color='g', linestyle='dashed') 

Create line that best fits the data

In [10]:
data_fit = np.polyfit(a,b,1)
data_line = hv.Slope(data_fit[0],data_fit[1]).opts(linewidth=2, color='b', linestyle='dashed') 

Create scatter plot for FCc at two representative echo time pairs.

In [11]:
scat_datashaded = df.hvplot.scatter(x='C(TE1,TE2)',y='C(TE2,TE3)', 
                                    xlabel=r"$FC_{c}(TE_{1},TE_{2})$",
                                    ylabel=r"$FC_{c}(TE_{2},TE_{3})$",
                                    aspect='square', xlim=(-.1,.5), ylim=(-.1,.5), datashade=True, frame_width=400, fontscale=1.1).opts(title='(a) Real Data')

Put it all together and add text annotations

In [12]:
def add_text(plot, element):
    ax = plot.handles["axis"]

    # Non-BOLD Line text
    ax.text(
        0.3, 0.25, r"$y = 1 \cdot x + 0.0$",
        color="red", fontsize=12, rotation=45, ha="left", va="bottom"
    )

    # BOLD Line text
    ax.text(
        0.01, 0.20, r"$ y = %.1f \cdot x + 0.0$" % BOLD_slope,
        color="green", fontsize=12, ha="left", va="bottom", rotation=75
    )

    # Linear Fit text
    ax.text(
        0.14, 0.26, r"$ y = %.1f \cdot x + %.1f$" % (data_fit[0],data_fit[1]),
        color="blue", fontsize=12, ha="left", va="bottom", rotation=63
    )

In [13]:
panel_a = (scat_datashaded * nonBOLD_line * BOLD_line * data_line * zero_point).opts(hooks=[add_text])

***
# 3. Create Supp. Figure 01 - Panel b: LOWESS Plot

In [14]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.formula.api import ols
import statsmodels.api as sm
import statsmodels.stats.diagnostic as diag

X = a
# Add a constant for the intercept
X_const = sm.add_constant(X)
model = sm.OLS(b, X_const).fit()

# Run Breusch-Pagan test
# The function takes residuals and design matrix (exog)
test_stat, p_value, _, _ = het_breuschpagan(model.resid, model.model.exog)

print(f"Breusch-Pagan test statistic: {test_stat:.3f}")
print(f"P-value: {p_value:.3f}")


Breusch-Pagan test statistic: 1923.922
P-value: 0.000


In [15]:
df2=pd.DataFrame([model.fittedvalues,model.resid], index=['Fitted','Residuals']).T

In [16]:
from statsmodels.nonparametric.smoothers_lowess import lowess

# Compute LOWESS smoother
smoothed = lowess(df2['Residuals'], df2['Fitted'], frac=0.3)
smooth_df = pd.DataFrame(smoothed, columns=['Fitted', 'Residuals'])

# Scatter plot of residuals
scatter = df2.hvplot.scatter(x='Fitted', y='Residuals', alpha=0.7, color='k', datashade=True, ylim=(-.1,.2), xlim=(-.1,.2), fontscale=1.1, frame_width=650)

# Smoothed curve
smooth_curve = smooth_df.hvplot.line(x='Fitted', y='Residuals', color='red')

# Horizontal zero line
zero_line = hv.HLine(0).opts(color='black', linestyle='dashed', linewidth=1.0)

# Combine plots
panel_b = (scatter * smooth_curve * zero_line).opts(
    title='(b) Fitted Values vs. Residuals & LOWESS line',
    xlabel='Fitted values', ylabel='Residuals')

In [31]:
bold_banner = pn.pane.HTML(
    """
    <div style="
        width:100%;
        background:#000;
        color:#fff;
        font-weight:700;
        text-align:center;
        padding:10px 12px;
        box-sizing:border-box;
        font-size:24px;
        width:1675px;
    ">
      Issue with quantification via linear fit
    </div>
    """,
    sizing_mode="scale_width",
)

In [33]:
SuppFig01 = pn.Column(bold_banner, pn.Row(panel_a,panel_b))
SuppFig01.save('./figures/pBOLD_SuppFig01.html')

Here is the final suppl. figure 01 in static form (for github rendering)

![Suppl. Figure 01](figures/pBOLD_SuppFig01.png)